# StatPitch
## Notebook 4 — FastAPI Backend

---

We turn the trained models into a **REST API** — a URL any app or browser can call to get predictions.

**What this notebook builds:**

```
statpitch_api/
├── main.py            ← FastAPI app  (the server)
├── predictor.py       ← loads models, builds predictions
├── requirements.txt   ← Python libraries list for deployment
├── models/
│   ├── model_xgb_1x2.pkl
│   ├── model_home_goals.pkl
│   └── model_away_goals.pkl
└── data/
    ├── model_config.json
    └── team_stats.json   ← current stats per team (built here)
```

**API endpoints we expose:**

| Method | Endpoint | Returns |
|---|---|---|
| GET | `/health` | API status |
| GET | `/teams` | All teams ranked by Elo |
| POST | `/predict` | Full market prediction (JSON body) |
| GET | `/predict/{home}/{away}` | Same, via URL (easy to test in browser) |
| GET | `/docs` | Auto-generated interactive documentation |


---
## Step 1 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

SAVE_DIR    = '/content/drive/MyDrive/StatPitch'
PROJECT_DIR = os.path.join(SAVE_DIR, 'statpitch_api')

os.chdir(SAVE_DIR)
print(f'Working directory: {os.getcwd()}')
print(f'Project will be built at: {PROJECT_DIR}')

In [ ]:
# Install FastAPI and Uvicorn (the server that runs FastAPI)
!pip install fastapi uvicorn[standard] --quiet
print('FastAPI and Uvicorn installed!')

---
## Step 2 — Build current team statistics

The API needs to look up each team's current form and Elo rating instantly.
We precompute this once from the full dataset and save it as `team_stats.json`.

**Why not use the rolling stats from Notebook 02?** Those stats had `.shift(1)` applied, so they represent a team's form *before* their last match. Here we include the last match to get truly current stats.

In [ ]:
import pandas as pd
import numpy as np
import json

print('Loading data...')
df = pd.read_csv('football_data_clean.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

elo_df  = pd.read_csv('current_elo_ratings.csv')
elo_map = dict(zip(elo_df['team'], elo_df['elo']))

print(f'Matches loaded: {len(df):,}')
print(f'Teams with Elo ratings: {len(elo_map)}')

In [ ]:
# Create long format: one row per team per match
home_r = df[['date', 'home_team', 'home_score', 'away_score', 'result']].copy()
home_r.columns = ['date', 'team', 'goals_scored', 'goals_conceded', 'result']
home_r['won']  = (home_r['result'] == 'home_win').astype(int)
home_r['drew'] = (home_r['result'] == 'draw').astype(int)

away_r = df[['date', 'away_team', 'away_score', 'home_score', 'result']].copy()
away_r.columns = ['date', 'team', 'goals_scored', 'goals_conceded', 'result']
away_r['won']  = (away_r['result'] == 'away_win').astype(int)
away_r['drew'] = (away_r['result'] == 'draw').astype(int)

all_r = pd.concat([home_r, away_r]).sort_values(['team', 'date']).reset_index(drop=True)
all_r['points'] = all_r['won'] * 3 + all_r['drew']

# Compute current stats per team (last 5 and 10 matches, INCLUDING the last match)
team_stats = {}
for team, grp in all_r.groupby('team'):
    r5  = grp.tail(5)
    r10 = grp.tail(10)
    team_stats[team] = {
        'elo':          round(float(elo_map.get(team, 1500)), 2),
        'gs_avg5':      round(float(r5['goals_scored'].mean()),   3),
        'gc_avg5':      round(float(r5['goals_conceded'].mean()), 3),
        'pts_avg5':     round(float(r5['points'].mean()),         3),
        'win_rate5':    round(float(r5['won'].mean()),            3),
        'gs_avg10':     round(float(r10['goals_scored'].mean()),  3),
        'gc_avg10':     round(float(r10['goals_conceded'].mean()),3),
        'pts_avg10':    round(float(r10['points'].mean()),        3),
        'win_rate10':   round(float(r10['won'].mean()),           3),
        'last_match':   str(grp['date'].max().date()),
        'total_matches':int(len(grp)),
    }

with open('team_stats.json', 'w') as f:
    json.dump(team_stats, f, indent=2)

print(f'team_stats.json saved — {len(team_stats)} teams')
print()
print('Top 10 teams by current Elo:')
top10 = sorted(team_stats.items(), key=lambda x: x[1]['elo'], reverse=True)[:10]
for t, s in top10:
    print(f'  {t:<25}  Elo: {s["elo"]:.0f}   Win rate (last 5): {s["win_rate5"]*100:.0f}%')

---
## Step 3 — Create project folder and copy model files

In [ ]:
import shutil

# Create folder structure
os.makedirs(os.path.join(PROJECT_DIR, 'models'), exist_ok=True)
os.makedirs(os.path.join(PROJECT_DIR, 'data'),   exist_ok=True)

# Copy trained model files
for fname in ['model_xgb_1x2.pkl', 'model_home_goals.pkl', 'model_away_goals.pkl']:
    shutil.copy2(os.path.join(SAVE_DIR, fname), os.path.join(PROJECT_DIR, 'models', fname))
    print(f'Copied  models/{fname}')

# Copy data files
for fname in ['model_config.json', 'team_stats.json']:
    shutil.copy2(os.path.join(SAVE_DIR, fname), os.path.join(PROJECT_DIR, 'data', fname))
    print(f'Copied  data/{fname}')

print()
print(f'Project directory ready: {PROJECT_DIR}')

---
## Step 4 — Write `predictor.py`

This module loads the models and does all the prediction logic. `main.py` calls it — it knows nothing about HTTP, it just returns Python dicts.

In [ ]:
%%writefile /content/drive/MyDrive/StatPitch/statpitch_api/predictor.py
# predictor.py  --  loads models and generates predictions for all markets
import joblib
import json
import numpy as np
from scipy.stats import poisson
from pathlib import Path

BASE = Path(__file__).parent

# ─── Load everything once at import time, not per request ────────────────────
_xgb  = joblib.load(BASE / 'models' / 'model_xgb_1x2.pkl')
_home = joblib.load(BASE / 'models' / 'model_home_goals.pkl')
_away = joblib.load(BASE / 'models' / 'model_away_goals.pkl')

with open(BASE / 'data' / 'model_config.json') as f:
    _cfg = json.load(f)

with open(BASE / 'data' / 'team_stats.json') as f:
    TEAM_STATS = json.load(f)

FEATURE_COLS = _cfg['feature_cols']
ELO_DEFAULT  = float(_cfg.get('elo_start', 1500))

# Default values used when a team is not in the dataset
_AVG = {
    'elo': ELO_DEFAULT,
    'gs_avg5': 1.30,  'gc_avg5': 1.30,  'pts_avg5': 1.20,  'win_rate5': 0.35,
    'gs_avg10': 1.30, 'gc_avg10': 1.30, 'pts_avg10': 1.20, 'win_rate10': 0.35,
}


def get_teams():
    return sorted(TEAM_STATS.keys())


def _build_features(home, away, is_neutral):
    h  = TEAM_STATS.get(home, _AVG)
    a  = TEAM_STATS.get(away, _AVG)
    he = h.get('elo', ELO_DEFAULT)
    ae = a.get('elo', ELO_DEFAULT)
    row = {
        'home_elo':        he,
        'away_elo':        ae,
        'elo_diff':        he - ae,
        'elo_prob_home':   1 / (1 + 10 ** ((ae - he) / 400)),
        'home_gs_avg5':    h.get('gs_avg5',    _AVG['gs_avg5']),
        'home_gc_avg5':    h.get('gc_avg5',    _AVG['gc_avg5']),
        'home_pts_avg5':   h.get('pts_avg5',   _AVG['pts_avg5']),
        'home_win_rate5':  h.get('win_rate5',  _AVG['win_rate5']),
        'home_gs_avg10':   h.get('gs_avg10',   _AVG['gs_avg10']),
        'home_gc_avg10':   h.get('gc_avg10',   _AVG['gc_avg10']),
        'home_pts_avg10':  h.get('pts_avg10',  _AVG['pts_avg10']),
        'home_win_rate10': h.get('win_rate10', _AVG['win_rate10']),
        'away_gs_avg5':    a.get('gs_avg5',    _AVG['gs_avg5']),
        'away_gc_avg5':    a.get('gc_avg5',    _AVG['gc_avg5']),
        'away_pts_avg5':   a.get('pts_avg5',   _AVG['pts_avg5']),
        'away_win_rate5':  a.get('win_rate5',  _AVG['win_rate5']),
        'away_gs_avg10':   a.get('gs_avg10',   _AVG['gs_avg10']),
        'away_gc_avg10':   a.get('gc_avg10',   _AVG['gc_avg10']),
        'away_pts_avg10':  a.get('pts_avg10',  _AVG['pts_avg10']),
        'away_win_rate10': a.get('win_rate10', _AVG['win_rate10']),
        'h2h_home_win_rate': 0.40,
        'h2h_avg_goals':     2.50,
        'h2h_num_games':     0,
        'is_neutral':        int(is_neutral),
        'tournament_weight': 1.00,
    }
    return np.array([[row[c] for c in FEATURE_COLS]])


def _markets(lh, la, max_g=9):
    mat = np.outer(
        [poisson.pmf(i, lh) for i in range(max_g)],
        [poisson.pmf(j, la) for j in range(max_g)],
    )
    idx  = np.array([[i + j for j in range(max_g)] for i in range(max_g)])
    btts = float((1 - poisson.pmf(0, lh)) * (1 - poisson.pmf(0, la)))
    top  = sorted(
        [{'score': f'{i}-{j}', 'probability': round(float(mat[i, j]), 4)}
         for i in range(max_g) for j in range(max_g)],
        key=lambda x: x['probability'],
        reverse=True,
    )[:10]
    return {
        'match_result': {
            'home_win': round(float(np.sum(np.tril(mat, -1))), 4),
            'draw':     round(float(np.sum(np.diag(mat))),     4),
            'away_win': round(float(np.sum(np.triu(mat, 1))),  4),
        },
        'over_under': {
            'over_1_5': round(float(np.sum(mat[idx > 1])), 4),
            'over_2_5': round(float(np.sum(mat[idx > 2])), 4),
            'over_3_5': round(float(np.sum(mat[idx > 3])), 4),
        },
        'btts': {'yes': round(btts, 4), 'no': round(1 - btts, 4)},
        'correct_score': top,
    }


def predict(home, away, is_neutral=True):
    feat  = _build_features(home, away, is_neutral)
    lh    = float(_home.predict(feat)[0])
    la    = float(_away.predict(feat)[0])
    xgb_p = _xgb.predict_proba(feat)[0]   # [away_prob, draw_prob, home_prob]
    out   = _markets(lh, la)
    mr    = out['match_result']
    # Blend XGBoost (60%) + Poisson (40%) for 1X2
    out['match_result'] = {
        'home_win': round(0.6 * xgb_p[2] + 0.4 * mr['home_win'], 4),
        'draw':     round(0.6 * xgb_p[1] + 0.4 * mr['draw'],     4),
        'away_win': round(0.6 * xgb_p[0] + 0.4 * mr['away_win'], 4),
    }
    h = TEAM_STATS.get(home, _AVG)
    a = TEAM_STATS.get(away, _AVG)
    return {
        'home_team': home,
        'away_team': away,
        'expected_goals': {'home': round(lh, 3), 'away': round(la, 3)},
        'team_info': {
            'home_elo': round(h.get('elo', ELO_DEFAULT), 1),
            'away_elo': round(a.get('elo', ELO_DEFAULT), 1),
            'elo_diff': round(h.get('elo', ELO_DEFAULT) - a.get('elo', ELO_DEFAULT), 1),
        },
        **out,
    }

---
## Step 5 — Write `main.py`

The FastAPI app. It receives HTTP requests, validates inputs, calls `predictor.py`, and returns JSON responses. The `@app.get` and `@app.post` decorators define the endpoints.

In [ ]:
%%writefile /content/drive/MyDrive/StatPitch/statpitch_api/main.py
# main.py  --  FastAPI application entry point
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional
import predictor

app = FastAPI(
    title='World Cup Predictor API',
    description='Predict match outcomes for international football using ML',
    version='1.0.0',
)

# CORS: allows any frontend (React, HTML) to call this API
app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_methods=['*'],
    allow_headers=['*'],
)


class PredictRequest(BaseModel):
    home_team:  str
    away_team:  str
    is_neutral: Optional[bool] = True


def _validate(home: str, away: str):
    teams = predictor.get_teams()
    if home not in teams:
        raise HTTPException(status_code=404, detail=f'Team not found: {home}')
    if away not in teams:
        raise HTTPException(status_code=404, detail=f'Team not found: {away}')
    if home == away:
        raise HTTPException(status_code=400, detail='Teams must be different')


@app.get('/')
def root():
    return {
        'service':   'World Cup Predictor API',
        'version':   '1.0.0',
        'endpoints': ['/health', '/teams', '/predict', '/docs'],
    }


@app.get('/health')
def health():
    return {
        'status':          'ok',
        'teams_available': len(predictor.TEAM_STATS),
    }


@app.get('/teams')
def get_teams():
    ranked = sorted(
        [{'team': k, 'elo': round(v.get('elo', 1500), 1)}
         for k, v in predictor.TEAM_STATS.items()],
        key=lambda x: x['elo'],
        reverse=True,
    )
    return {'count': len(ranked), 'teams': ranked}


@app.post('/predict')
def predict_post(req: PredictRequest):
    _validate(req.home_team, req.away_team)
    return predictor.predict(req.home_team, req.away_team, req.is_neutral)


@app.get('/predict/{home_team}/{away_team}')
def predict_get(home_team: str, away_team: str, is_neutral: bool = True):
    _validate(home_team, away_team)
    return predictor.predict(home_team, away_team, is_neutral)

---
## Step 6 — Write `requirements.txt`

In [ ]:
%%writefile /content/drive/MyDrive/StatPitch/statpitch_api/requirements.txt
fastapi>=0.110.0
uvicorn[standard]>=0.29.0
xgboost>=2.0.0
scikit-learn>=1.4.0
numpy>=1.26.0
scipy>=1.13.0
joblib>=1.4.0
pandas>=2.0.0

---
## Step 7 — Test the predictor locally

Before deploying, we verify `predictor.py` works correctly by importing it directly in Colab.

In [ ]:
import sys

# Point Python to the project folder so it can find predictor.py
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)   # predictor.py uses Path(__file__).parent to find models

# Import and test
import importlib
import predictor
importlib.reload(predictor)

print(f'Loaded {len(predictor.TEAM_STATS)} teams')
print(f'Feature columns: {len(predictor.FEATURE_COLS)}')
print()

# Sanity check: top 5 teams by Elo
top5 = sorted(predictor.TEAM_STATS.items(), key=lambda x: x[1]['elo'], reverse=True)[:5]
print('Top 5 teams by current Elo:')
for team, stats in top5:
    print(f'  {team:<25}  Elo: {stats["elo"]:.0f}')

In [ ]:
# Test 1: Brazil vs Argentina
result = predictor.predict('Brazil', 'Argentina', is_neutral=True)
print('Brazil vs Argentina:')
print(json.dumps(result, indent=2))

In [ ]:
# Test 2: France vs England
result2 = predictor.predict('France', 'England', is_neutral=True)

mr = result2['match_result']
ou = result2['over_under']
bt = result2['btts']
eg = result2['expected_goals']

print('France vs England — formatted output:')
print(f'  Expected goals:  France {eg["home"]:.2f}  —  England {eg["away"]:.2f}')
print(f'  1X2:  France {mr["home_win"]*100:.1f}%   Draw {mr["draw"]*100:.1f}%   England {mr["away_win"]*100:.1f}%')
print(f'  Over 2.5:  {ou["over_2_5"]*100:.1f}%   Under 2.5: {(1-ou["over_2_5"])*100:.1f}%')
print(f'  BTTS Yes:  {bt["yes"]*100:.1f}%   BTTS No:   {bt["no"]*100:.1f}%')
print()
print('  Top 5 scorelines:')
for s in result2['correct_score'][:5]:
    print(f'    {s["score"]}  ->  {s["probability"]*100:.1f}%')

In [ ]:
# Test 3: /teams endpoint logic
teams = predictor.get_teams()
print(f'Total teams available: {len(teams)}')
print(f'First 10 (alphabetical): {teams[:10]}')
print()

# Test error handling for unknown team
unknown = 'Unknown FC'
result_unknown = predictor.predict(unknown, 'Brazil')   # Should use defaults
print(f'Prediction for unknown team "{unknown}" vs Brazil (uses average defaults):')
print(f'  Elo info: {result_unknown["team_info"]}')
print(f'  1X2: {result_unknown["match_result"]}')

---
## Step 8 — Verify final project structure

In [ ]:
os.chdir(SAVE_DIR)   # Reset working directory

print('statpitch_api/')
for root, dirs, files in os.walk(PROJECT_DIR):
    level = root.replace(PROJECT_DIR, '').count(os.sep)
    indent = '    ' * level
    folder = os.path.basename(root)
    if level > 0:
        print(f'{indent}{folder}/')
    for file in sorted(files):
        sub = '    ' * (level + 1)
        size = os.path.getsize(os.path.join(root, file))
        size_str = f'{size/1024:.0f} KB' if size > 1024 else f'{size} B'
        print(f'{sub}{file}  ({size_str})')